# World Cities Gazetteer

Starter notebook for the Kaggle dataset in this folder. It loads `cities.csv` with pandas and walks through quality checks and a first piece of analysis.

**Files:** `cities.csv`, `data_dictionary.csv`.

## 1. Load the data

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("cities.csv", low_memory=False)
df["population"] = pd.to_numeric(df["population"], errors="coerce")
print(df.shape)

## 2. Shape, types and missing values

In [ ]:
print(f"{df.shape[0]:,} rows x {df.shape[1]} columns")

missing = df.isna().mean().mul(100).sort_values(ascending=False)
missing[missing > 0].round(1).to_frame("missing_%").head(15)

## 3. Numeric summary

In [ ]:
df.describe(include='number').T.round(2).head(20)

## 4. Zipf's law

Rank versus population on log axes — a straight line means a Zipfian urban system.

In [ ]:
big = df[df["population"] > 0].sort_values("population", ascending=False)
big = big[big["country_code"] == "USA"].reset_index(drop=True)
big["rank"] = big.index + 1
ax = big.plot.scatter(x="rank", y="population", loglog=True, figsize=(8, 5))

## 5. Primacy

Largest city's share of national population, top 20 countries.

In [ ]:
prim = (df.groupby("country_name")["population_share_country_pct"]
        .max().dropna().sort_values(ascending=False).head(20))
prim.plot.barh(figsize=(8, 6))

## 6. Megacities

In [ ]:
mega = df[df["population"] >= 5_000_000].nlargest(20, "population")
mega[["name", "country_name", "population", "population_class", "is_capital"]].set_index("name")

## 7. Elevation and settlement

In [ ]:
df.dropna(subset=["dem_m"]).query("-100 < dem_m < 5000").hist(
    column="dem_m", bins=60, figsize=(9, 4))

## 8. Country urban profiles

In [ ]:
prof = (df.groupby("country_name")
        .agg(cities=("geonameid", "count"), median_pop=("population", "median"),
             total_urban=("population", "sum"), density=("country_pop_density_per_km2", "first")))
prof.query("cities >= 20").sort_values("total_urban", ascending=False).head(15)

## Next steps

See `KAGGLE.md` for analysis ideas, caveats and provenance.